<a href="https://colab.research.google.com/github/Muqtarali/Generative-AI-using-LangChain/blob/main/tool_calling_project/currency_conveter_using_tool_calling_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["GROQ_API_KEY"]='gsk_5eIo'
os.environ["GOOGLE_API_KEY"]='AQ.Ab8'
os.environ["HUGGINGFACEHUB_API_TOKEN"]='hf_xk'

In [2]:
!pip install -q langchain-openai langchain-core requests langchain-groq

In [3]:
from langchain_openai import ChatOpenAI
from langchain_groq.chat_models import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
#Create tool
from langchain_core.tools import tool

@tool
def multiply(a: int,b: int)->int:
  """Given two number a and b this tool return their product"""
  return a*b


In [5]:
print(multiply.invoke({'a':4,'b':5}))
#

20


In [6]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given two number a and b this tool return their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


##Tool BInding

In [7]:
model = ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0.2,
)

In [8]:
llm_with_tool=model.bind_tools([multiply])

In [9]:
queary=HumanMessage(content='what is multipe of 2 with 5')

In [10]:
message=[queary]

In [11]:
print(llm_with_tool.invoke('hii'))

content='Hello! How can I help you today?' additional_kwargs={'reasoning_content': 'User says "hii". Likely greeting. We should respond politely.'} response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 130, 'total_tokens': 164, 'completion_time': 0.044848943, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.008846703, 'prompt_tokens_details': None, 'queue_time': 1.533747545, 'total_time': 0.053695646}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a17dcc-0897-7717-a4e1-be413c9b5ea1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 130, 'output_tokens': 34, 'total_tokens': 164, 'output_token_details': {'reasoning': 16}}


In [12]:
result=llm_with_tool.invoke(message)

In [13]:
message.append(result)

In [14]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 2, 'b': 5},
 'id': 'fc_a0410ee9-3213-4d2c-ab67-5cf442c24dbc',
 'type': 'tool_call'}

#Tool Execution

In [15]:
tool_result=multiply.invoke(result.tool_calls[0])

In [16]:
message.append(tool_result)

In [17]:
message

[HumanMessage(content='what is multipe of 2 with 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what is multipe of 2 with 5". Likely they mean "what is the multiple of 2 with 5" or "what is the product of 2 and 5". So we need to multiply 2 and 5. Use the multiply function.', 'tool_calls': [{'id': 'fc_a0410ee9-3213-4d2c-ab67-5cf442c24dbc', 'function': {'arguments': '{"a":2,"b":5}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 138, 'total_tokens': 225, 'completion_time': 0.098970847, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.007904022, 'prompt_tokens_details': None, 'queue_time': 0.077822296, 'total_time': 0.106874869}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_976a85d17a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run

In [18]:
llm_with_tool.invoke(message).content

'10'

#currency_conversion

In [19]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [20]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [21]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1793059201,
 'time_last_update_utc': 'Tue, 27 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1793145601,
 'time_next_update_utc': 'Wed, 28 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.2718}

In [22]:
convert.invoke({"base_currency_value": 10, "conversion_rate": 95.2718})

952.718

In [23]:
!pip install -U langchain-google-genai

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [25]:
#Tool buinding

model =ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

In [26]:
model_with_tool=model.bind_tools([get_conversion_factor,convert])

In [27]:
message=[]

In [28]:
query = HumanMessage(
    "Call BOTH tools in this turn. "
    "1. Call get_conversion_factor(base_currency='INR', target_currency='USD'). "
    "2. Simultaneously call convert(base_currency_value=10, conversion_rate=1.0) as a placeholder."
)

In [29]:
message.append(query)

In [30]:
message

[HumanMessage(content="Call BOTH tools in this turn. 1. Call get_conversion_factor(base_currency='INR', target_currency='USD'). 2. Simultaneously call convert(base_currency_value=10, conversion_rate=1.0) as a placeholder.", additional_kwargs={}, response_metadata={})]

In [31]:
#Tool caling
result=model_with_tool.invoke(message)

In [32]:
result

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"conversion_rate": 1, "base_currency_value": 10}'}, '__gemini_function_call_thought_signatures__': {'call_2225239': 'ErgjCrUjAWkUfRPbW5aBsFRga5D7qIAQlu/x14Y3gFweL3275y92qyHojlSYLzOKrTm+hMM+sVqzc/OWX2+WAMXpjqbDK8nVxuTdO/ITFpMeL5bl2nUHvMS3EonMrHvMWaFnNRJkaeJs7/qA9FW+YyQer6nkSDRkQl1HKsxiBCxO7Hgy1HxWb9CsUmbf0wbY7GBl5IsKhKPLSINtq9F/i5klhwSYbOvIn2s95ab/T9i4kPRbIp267sxgbBovMFnOZ4f6u++hiOHo2E2UI5ZmSalxNEXWMHZK2TXSzY45pjA3olslqDJBSTgMTZ73uGyEAgZsG+VdmdV9YKS80wYlHFRQHUQ3tWKd+rwjWVpHszY7R+m4hAorxJY5Afbv0j7kTTF7jNg3fWz7oHXKr+hKlTZTcB31D3B64hwXaiXAu2lYLWRZ7zYGApsIBqKYezXeNOY48Nc4O5a6xfS+iLjIlBEAig5yw49j600GWKWWtUIIsagRAaxyTZKzPsGPZczVQfiRkPb267IgNQ+GmSCdznwpYfWEKKy8UXIXWL3fI497Bmmx5+2V1tUJKfqAJjno5QSAr/MOSjKLvfw9TtQ2lHhU/1lH/AXpEIf0/bik017UpbgoOiHKJGYgcUGShKT43TXTR+9AnB5pNZATJDq0wIIPIv69Eu5t5axenPmC+GfKZrLoxVZX2cRcH/rDouXnVgA2QohoXgTd3cejy2SMDtW5Uy0IFZqkhHcCu1FIE80veM9bmzYINWo7Tgehpy8VglnbRHen0HeeRMHmWw1Amy

In [33]:
result.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'USD', 'base_currency': 'INR'},
  'id': 'call_2225239',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'conversion_rate': 1, 'base_currency_value': 10},
  'id': 'call_2225240',
  'type': 'tool_call'}]

In [34]:
import json

for tool_call in result.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    message.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    message.append(tool_message2)



In [36]:
message

[HumanMessage(content="Call BOTH tools in this turn. 1. Call get_conversion_factor(base_currency='INR', target_currency='USD'). 2. Simultaneously call convert(base_currency_value=10, conversion_rate=1.0) as a placeholder.", additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1793059201, "time_last_update_utc": "Tue, 27 Oct 2026 00:00:01 +0000", "time_next_update_unix": 1793145601, "time_next_update_utc": "Wed, 28 Oct 2026 00:00:01 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.0105}', name='get_conversion_factor', tool_call_id='call_2225239'),
 ToolMessage(content='0.10500000000000001', name='convert', tool_call_id='call_2225240')]

In [37]:
model_with_tool.invoke(message).content

[]